[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import torch

In [33]:
# ✏️ YOUR IMPLEMENTATION HERE

# def cross_entropy_loss(logits, targets):
#     print('logits shape:', logits.shape)
#     print('targets shape:', targets.shape)
#     return torch.logsumexp(logits, dim=-1) - logits[:, targets]
#     pass  # log_probs = logits - logsumexp(...)

def cross_entropy_loss(logits, targets):
    print('logits shape:', logits.shape)
    print('targets shape:', targets.shape)

    # Cross-entropy needs one loss per sample:
    # 1) logsumexp must reduce over the class dimension (dim=1), not the whole tensor.
    # 2) logits[:, targets] is wrong: it selects all target columns for every row -> (B, B).
    # 3) Pair each row index with its own target index to get one correct-class logit per sample -> (B,).
    logsumexp = torch.logsumexp(logits, dim=1)
    target_logits = logits[torch.arange(logits.shape[0]), targets]
    loss = logsumexp - target_logits

    return loss.mean()

In [34]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

logits shape: torch.Size([4, 10])
targets shape: torch.Size([4])
Loss: tensor(2.4274)
Ref:  tensor(2.4274)


In [35]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
logits shape: torch.Size([4, 10])
targets shape: torch.Size([4])
  ✅ [1/4] Matches F.cross_entropy (5.7ms)
logits shape: torch.Size([2, 3])
targets shape: torch.Size([2])
  ✅ [2/4] Numerical stability (0.6ms)
logits shape: torch.Size([8, 5])
targets shape: torch.Size([8])
  ✅ [3/4] Scalar output (0.3ms)
logits shape: torch.Size([8, 5])
targets shape: torch.Size([8])
  ✅ [4/4] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (7.3ms total)
  Progress saved. Run status() to see your dashboard.

